# Final Submission — Qwen2.5-7B + BaselinePrompt (fully offline)
Uses the competition's own BaselinePrompt to extract SDRF metadata from test papers.
Runs fully offline using the `ragnar123/qwen2-5-7b-instruct` Kaggle dataset.

**To use:** Attach `ragnar123/qwen2-5-7b-instruct` dataset to this notebook before running.

## 1. Imports and paths

In [1]:
import json, re, warnings
from pathlib import Path
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
warnings.filterwarnings('ignore')

# ── Environment detection ────────────────────────────────────────────
IS_KAGGLE = Path('/kaggle').exists()

if IS_KAGGLE:
    MODEL_ID         = str(Path('/kaggle/input/qwen2-5-7b-instruct'))   # offline 7B
    DATA_PATH        = Path('/kaggle/input/harmonizing-the-data-of-your-data')
    OUT_PATH         = Path('/kaggle/working/submission_final.csv')
    _MAX_NEW_TOKENS  = 2048
    _WORD_LIMIT      = 4000
else:
    ROOT             = Path.cwd().parent
    MODEL_ID         = 'Qwen/Qwen2.5-1.5B-Instruct'   # cached, fits 6 GB VRAM
    DATA_PATH        = ROOT / 'data'
    OUT_PATH         = ROOT / 'outputs' / 'submission_final.csv'
    _MAX_NEW_TOKENS  = 512    # smaller KV cache budget for 6 GB VRAM
    _WORD_LIMIT      = 1500   # keep input short to avoid OOM

SAMPLE_SUB  = DATA_PATH / 'SampleSubmission.csv'

# Probe all known variants of the PubText directory name on Kaggle
_pubtext_candidates = [
    DATA_PATH / 'Test_PubText' / 'Test PubText',
    DATA_PATH / 'Test_PubText',
    DATA_PATH / 'TestPubText',
    DATA_PATH / 'Test PubText',
]
PUBTEXT_DIR = next((p for p in _pubtext_candidates if p.exists()), _pubtext_candidates[0])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device      : {device}')
if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU         : {torch.cuda.get_device_name(0)}  ({total:.1f} GB VRAM)')
print(f'Model ID    : {MODEL_ID}')
print(f'PubText dir : {PUBTEXT_DIR} — exists: {PUBTEXT_DIR.exists()}')

pub_files = list(PUBTEXT_DIR.glob('*.json')) if PUBTEXT_DIR.exists() else []
print(f'PubText .json files found: {len(pub_files)}')
for f in sorted(pub_files)[:5]:
    print(f'  {f.name}')

c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device      : cuda
GPU         : NVIDIA GeForce GTX 1060  (6.4 GB VRAM)
Model ID    : Qwen/Qwen2.5-1.5B-Instruct
PubText dir : c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\data\TestPubText — exists: True
PubText .json files found: 16
  PubText.json
  PXD004010_PubText.json
  PXD016436_PubText.json
  PXD019519_PubText.json
  PXD025663_PubText.json


## 2. Load SampleSubmission — extract raw filenames per PXD

In [2]:
sample_sub = pd.read_csv(SAMPLE_SUB, dtype=str)
print(f'Sample submission shape: {sample_sub.shape}')
print(f'Columns: {sample_sub.columns[:5].tolist()} ...')
print()

# Group raw filenames by PXD
pxd_to_raws: dict[str, list[str]] = {}
for _, row in sample_sub.iterrows():
    pxd = row['PXD']
    raw = row['Raw Data File']
    if pxd not in pxd_to_raws:
        pxd_to_raws[pxd] = []
    pxd_to_raws[pxd].append(raw)

print('Test PXDs and raw file counts:')
for pxd, raws in sorted(pxd_to_raws.items()):
    print(f'  {pxd}: {len(raws)} raw files')

Sample submission shape: (1659, 81)
Columns: ['ID', 'PXD', 'Raw Data File', 'Characteristics[Age]', 'Characteristics[AlkylationReagent]'] ...

Test PXDs and raw file counts:
  PXD004010: 10 raw files
  PXD016436: 18 raw files
  PXD019519: 6 raw files
  PXD025663: 12 raw files
  PXD040582: 24 raw files
  PXD050621: 9 raw files
  PXD061009: 2 raw files
  PXD061090: 6 raw files
  PXD061136: 2 raw files
  PXD061195: 1376 raw files
  PXD061285: 60 raw files
  PXD062014: 24 raw files
  PXD062469: 32 raw files
  PXD062877: 48 raw files
  PXD064564: 30 raw files


## 3. Load PubText JSONs for test PXDs

In [3]:
SECTIONS_TO_KEEP = ['TITLE', 'ABSTRACT', 'INTRO', 'METHODS',
                    'RESULTS', 'DISCUSS', 'FIG', 'SUPPL']

def load_pubtext(pxd: str) -> str:
    # Try JSON first (all known naming variants), then .txt as last resort
    json_candidates = [
        PUBTEXT_DIR / f'{pxd}_PubText.json',
        PUBTEXT_DIR / f'{pxd}.json',
    ]
    txt_candidates = [
        PUBTEXT_DIR / f'{pxd}_PubText.txt',
        PUBTEXT_DIR / f'{pxd}.txt',
    ]
    for path in json_candidates:
        if path.exists():
            payload = json.loads(path.read_text(encoding='utf-8'))
            if isinstance(payload, dict):
                parts = []
                for sec in SECTIONS_TO_KEEP:
                    val = payload.get(sec, '')
                    if val and isinstance(val, str):
                        parts.append(f'[{sec}]\n{val.strip()}')
                if not parts:
                    parts = [str(v) for v in payload.values()
                             if isinstance(v, str) and v.strip()]
                return '\n\n'.join(parts)
            elif isinstance(payload, list):
                return ' '.join(str(x) for x in payload)
            else:
                return str(payload)
    # Fallback: plain text file
    for path in txt_candidates:
        if path.exists():
            return path.read_text(encoding='utf-8', errors='replace').strip()
    return ''

pxd_texts: dict[str, str] = {}
for pxd in sorted(pxd_to_raws.keys()):
    text = load_pubtext(pxd)
    pxd_texts[pxd] = text
    status = f'{len(text):,} chars' if text else 'NOT FOUND'
    print(f'  {pxd}: {status}')

  PXD004010: 25,561 chars
  PXD016436: 7,684 chars
  PXD019519: 84,689 chars
  PXD025663: 46,369 chars
  PXD040582: 82,627 chars
  PXD050621: 48,738 chars
  PXD061009: 90,587 chars
  PXD061090: 42,103 chars
  PXD061136: 49,726 chars
  PXD061195: 35,843 chars
  PXD061285: 74,904 chars
  PXD062014: 75,415 chars
  PXD062469: 34,390 chars
  PXD062877: 77,663 chars
  PXD064564: 59,932 chars


## 4. Load Qwen2.5-7B-Instruct (offline)

In [4]:
from transformers import pipeline as hf_pipeline

print(f'Loading tokenizer from {MODEL_ID} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

print('Loading model...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
    device_map='auto',
    trust_remote_code=True,
)
model.eval()

pipe = hf_pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=_MAX_NEW_TOKENS,
    do_sample=False,
    temperature=None,
    top_p=None,
    return_full_text=False,
)
print('Model + pipeline ready.')

if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {used:.1f} GB used / {total:.1f} GB total')

Loading tokenizer from Qwen/Qwen2.5-1.5B-Instruct ...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading model...


Loading weights: 100%|██████████| 338/338 [00:05<00:00, 58.01it/s]
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'top_p', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Model + pipeline ready.
VRAM: 3.1 GB used / 6.4 GB total


## 5. Baseline prompt (from competition's BaselinePrompt.txt)

In [5]:
BASELINE_PROMPT = """You are extracting SDRF-Proteomics metadata from a scientific manuscript AND a provided list of MS .raw filenames.

PRIMARY GOAL
- Produce per-file (per .raw) SDRF-style metadata annotations using ONLY the metadata categories defined below.
- Extract ONLY exact text spans verbatim from the manuscript target sections, and/or exact substrings from provided .raw filenames when the filename token itself is explicit and self-descriptive.

INPUTS YOU WILL RECEIVE
A) MANUSCRIPT_TEXT: full manuscript text (may include section headers)
B) RAW_FILES: list of .raw file names (strings)

SDRF STRUCTURE
- One SDRF row represents one sample-data file relationship.

METADATA CATEGORIES (ONLY THESE):
Characteristics[Age], Characteristics[AlkylationReagent], Characteristics[AnatomicSiteTumor],
Characteristics[AncestryCategory], Characteristics[BMI], Characteristics[Bait],
Characteristics[BiologicalReplicate], Characteristics[CellLine], Characteristics[CellPart],
Characteristics[CellType], Characteristics[CleavageAgent], Characteristics[Compound],
Characteristics[ConcentrationOfCompound], Characteristics[Depletion],
Characteristics[DevelopmentalStage], Characteristics[Disease], Characteristics[DiseaseTreatment],
Characteristics[GeneticModification], Characteristics[Genotype], Characteristics[GrowthRate],
Characteristics[Label], Characteristics[MaterialType], Characteristics[Modification],
Characteristics[NumberOfBiologicalReplicates], Characteristics[NumberOfSamples],
Characteristics[NumberOfTechnicalReplicates], Characteristics[Organism],
Characteristics[OrganismPart], Characteristics[OriginSiteDisease], Characteristics[PooledSample],
Characteristics[ReductionReagent], Characteristics[SamplingTime], Characteristics[Sex],
Characteristics[Specimen], Characteristics[SpikedCompound], Characteristics[Staining],
Characteristics[Strain], Characteristics[SyntheticPeptide], Characteristics[TechnicalReplicate],
Characteristics[Temperature], Characteristics[Time], Characteristics[Treatment],
Characteristics[TumorCellularity], Characteristics[TumorGrade], Characteristics[TumorSize],
Characteristics[TumorSite], Characteristics[TumorStage],
Comment[AcquisitionMethod], Comment[CollisionEnergy], Comment[EnrichmentMethod],
Comment[FlowRateChromatogram], Comment[FractionIdentifier], Comment[FractionationMethod],
Comment[FragmentMassTolerance], Comment[FragmentationMethod], Comment[GradientTime],
Comment[Instrument], Comment[IonizationType], Comment[MS2MassAnalyzer],
Comment[NumberOfFractions], Comment[NumberOfMissedCleavages], Comment[PrecursorMassTolerance],
Comment[Separation],
FactorValue[Bait], FactorValue[CellPart], FactorValue[Compound],
FactorValue[ConcentrationOfCompound], FactorValue[Disease], FactorValue[FractionIdentifier],
FactorValue[GeneticModification], FactorValue[Temperature], FactorValue[Treatment]

ANNOTATION SCOPE: Extract ONLY from Title, Abstract, Methods/Materials sections.

FILENAME-AWARE MAPPING: Parse each .raw filename for fraction IDs, replicate markers,
condition markers, labels, timepoints. Use these to assign per-file metadata.

OUTPUT FORMAT: Return exactly one JSON object.
- Top level: each key is the exact .raw filename.
- Value: dict where keys are SDRF headers and values are lists of verbatim text spans.
- Required keys per file: "Source Name", "Assay Name", "Raw Data File"
- Output ONLY JSON. No commentary, no markdown fences."""

# Simpler flat-JSON prompt for smaller models (local 1.5B)
LOCAL_PROMPT = """You are an expert proteomics bioinformatician.
Given the text of a scientific publication, extract SDRF metadata fields.
Return ONLY a JSON object with the keys below. No explanation, no markdown fences.

Rules:
- Characteristics[Organism]: species studied (e.g. "Homo sapiens", "Mus musculus")
- Characteristics[OrganismPart]: tissue or biofluid (e.g. "blood plasma", "liver", "cell culture")
- Characteristics[Disease]: disease/condition (e.g. "colorectal cancer", "normal")
- Characteristics[CellType]: specific cell line or type (e.g. "HeLa", "T cell"). "Not Applicable" if none.
- Characteristics[Sex]: "male", "female", "mixed", or "Not Applicable"
- Characteristics[DevelopmentalStage]: "adult", "embryo", etc. "Not Applicable" if not mentioned.
- Characteristics[Modification]: list ALL PTMs in SDRF format NT=<name>;AC=UNIMOD:<id>;TA=<res>;MT=<Fixed|Variable>
  e.g. ["NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed", "NT=Oxidation;AC=UNIMOD:21;TA=M;MT=Variable"]
  Use ["Not Applicable"] if none mentioned.
- Characteristics[CleavageAgent]: enzyme (e.g. "Trypsin"). "Not Applicable" if not mentioned.
- Characteristics[Label]: labeling strategy (e.g. "TMT10", "label free sample")
- Comment[Instrument]: mass spectrometer model (e.g. "NT=Q Exactive HF;AC=MS:1002523")
- Comment[FragmentationMethod]: e.g. "NT=HCD;AC=MS:1000422"
- Comment[PrecursorMassTolerance]: e.g. "10 ppm"
- Comment[FragmentMassTolerance]: e.g. "0.02 Da"

Return exactly this JSON structure (all keys required, use "Not Applicable" for missing values):
{
  "Characteristics[Organism]": "species",
  "Characteristics[OrganismPart]": "tissue",
  "Characteristics[Disease]": "disease or normal",
  "Characteristics[CellType]": "cell type or Not Applicable",
  "Characteristics[Sex]": "male/female/mixed/Not Applicable",
  "Characteristics[DevelopmentalStage]": "stage or Not Applicable",
  "Characteristics[Modification]": ["NT=..."],
  "Characteristics[CleavageAgent]": "enzyme or Not Applicable",
  "Characteristics[Label]": "label or Not Applicable",
  "Comment[Instrument]": "NT=...;AC=... or Not Applicable",
  "Comment[FragmentationMethod]": "NT=...;AC=... or Not Applicable",
  "Comment[PrecursorMassTolerance]": "value or Not Applicable",
  "Comment[FragmentMassTolerance]": "value or Not Applicable"
}"""

# Word budget: 4000 on Kaggle (128k context), 1500 locally (fits in 3GB VRAM headroom)
_WORD_LIMIT = 4000 if IS_KAGGLE else 1500

def build_messages(pub_text: str, raw_files: list[str]) -> list[dict]:
    truncated = ' '.join(pub_text.split()[:_WORD_LIMIT])
    if IS_KAGGLE:
        raw_list = '\n'.join(raw_files)
        user_content = f'MANUSCRIPT_TEXT:\n{truncated}\n\nRAW_FILES:\n{raw_list}'
        return [
            {'role': 'system', 'content': BASELINE_PROMPT},
            {'role': 'user',   'content': user_content},
        ]
    else:
        return [
            {'role': 'system', 'content': LOCAL_PROMPT},
            {'role': 'user',   'content': f'Extract SDRF metadata from this publication text:\n\n{truncated}'},
        ]

print('Prompt template ready.')
print(f'Running in {"Kaggle (BaselinePrompt)" if IS_KAGGLE else "local (flat prompt)"} mode.')
print(f'Word limit: {_WORD_LIMIT} | Prompt: {len(BASELINE_PROMPT if IS_KAGGLE else LOCAL_PROMPT)} chars')

Prompt template ready.
Running in local (flat prompt) mode.
Word limit: 1500 | Prompt: 2286 chars


## 6. Run inference — one call per PXD (~15 total)

In [6]:
def run_inference(messages: list[dict]) -> str:
    """Run generation via pipeline (handles CUDA memory automatically)."""
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    result = pipe(prompt)
    return result[0]['generated_text']

def parse_json_output(raw_text: str) -> dict:
    """Extract JSON from model output robustly."""
    text = re.sub(r'```json|```', '', raw_text).strip()
    start = text.find('{')
    if start == -1:
        return {}
    depth, end = 0, -1
    for i, ch in enumerate(text[start:], start):
        if ch == '{': depth += 1
        elif ch == '}': depth -= 1
        if depth == 0:
            end = i + 1
            break
    if end == -1:
        return {}
    try:
        return json.loads(text[start:end])
    except json.JSONDecodeError:
        fixed = re.sub(r',\s*([}\]])', r'\1', text[start:end])
        try:
            return json.loads(fixed)
        except:
            return {}

# Run inference for each PXD
pxd_extractions: dict[str, dict] = {}

for i, (pxd, raw_files) in enumerate(sorted(pxd_to_raws.items()), 1):
    pub_text = pxd_texts.get(pxd, '')
    if not pub_text:
        print(f'[{i:2d}/{len(pxd_to_raws)}] {pxd}: NO TEXT — skipping')
        pxd_extractions[pxd] = {}
        continue

    print(f'[{i:2d}/{len(pxd_to_raws)}] {pxd} ({len(raw_files)} files) ...', end=' ', flush=True)
    messages = build_messages(pub_text, raw_files)
    try:
        raw_output = run_inference(messages)
        parsed = parse_json_output(raw_output)
        pxd_extractions[pxd] = parsed
        keys_found = [k for k in parsed if parsed[k] not in ('Not Applicable', ['Not Applicable'])]
        print(f'OK — {len(parsed)} keys, {len(keys_found)} non-NA')
    except Exception as e:
        print(f'ERROR: {e}')
        pxd_extractions[pxd] = {}

print(f'\nDone. {sum(1 for v in pxd_extractions.values() if v)}/{len(pxd_extractions)} PXDs succeeded.')

[ 1/15] PXD004010 (10 files) ... 

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


OK — 0 keys, 0 non-NA
[ 2/15] PXD016436 (18 files) ... 

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


OK — 13 keys, 6 non-NA
[ 3/15] PXD019519 (6 files) ... 

...

OK — 0 keys, 0 non-NA
[14/15] PXD062877 (48 files) ... 

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


OK — 13 keys, 4 non-NA
[15/15] PXD064564 (30 files) ... 

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


OK — 0 keys, 0 non-NA

Done. 6/15 PXDs succeeded.


In [8]:
# ── Debug: inspect actual model output per PXD ───────────────────────
n_ok = sum(1 for v in pxd_extractions.values() if v)
print(f'{n_ok}/{len(pxd_extractions)} PXDs returned parseable JSON\n')

for pxd, result in sorted(pxd_extractions.items()):
    if not result:
        continue
    first_key = next(iter(result))
    val = result[first_key]
    is_flat = first_key.startswith('Characteristics') or first_key.startswith('Comment')
    fmt = 'flat' if is_flat else 'per_file'
    print(f'{pxd} [{fmt}] — {len(result)} keys')
    if fmt == 'flat':
        for k, v in list(result.items())[:5]:
            print(f'  {k}: {repr(str(v)[:80])}')
    else:
        print(f'  First file: {first_key!r}')
        if isinstance(val, dict):
            for k, v in list(val.items())[:5]:
                print(f'    {k}: {repr(str(v)[:60])}')

6/15 PXDs returned parseable JSON

PXD016436 [flat] — 13 keys
  Characteristics[Organism]: 'bovine'
  Characteristics[OrganismPart]: 'whey protein'
  Characteristics[Disease]: 'Not Applicable'
  Characteristics[CellType]: 'Not Applicable'
  Characteristics[Sex]: 'Not Applicable'
PXD019519 [flat] — 13 keys
  Characteristics[Organism]: 'Not Applicable'
  Characteristics[OrganismPart]: 'Not Applicable'
  Characteristics[Disease]: 'Not Applicable'
  Characteristics[CellType]: 'Not Applicable'
  Characteristics[Sex]: 'Not Applicable'
PXD061090 [flat] — 13 keys
  Characteristics[Organism]: 'Not Applicable'
  Characteristics[OrganismPart]: 'synovial tissue'
  Characteristics[Disease]: 'synovial fibrosis'
  Characteristics[CellType]: 'Not Applicable'
  Characteristics[Sex]: 'Not Applicable'
PXD061136 [flat] — 13 keys
  Characteristics[Organism]: 'Not Applicable'
  Characteristics[OrganismPart]: 'Embryo'
  Characteristics[Disease]: 'Not Applicable'
  Characteristics[CellType]: 'Not Applicable'


## 7. Layers 2 & 3 \u2014 Rule-based extractors + Training majority fallback

In [ ]:
# ── Layer 2: Rule-based extractors ────────────────────────────────────
import re as _re

def extract_replicate_count(text: str):
    """Extract biological replicate count from paper text."""
    text_lower = text.lower()
    if _re.search(r'\bquadruplicate\b', text_lower): return 4
    if _re.search(r'\btriplicate\b', text_lower): return 3
    if _re.search(r'\bduplicate\b', text_lower): return 2
    for pattern in [
        r'(\d+)\s*(?:independent\s+)?biological\s+replicat',
        r'biological\s+replicat\w*\s*[=:(]\s*(\d+)',
        r'(\d+)\s+replicat\w*\s+(?:per|of|for|each)',
        r'(?:performed|conducted|run)\s+(?:in\s+)?(\d+)\s+replicat',
        r'n\s*=\s*(\d+)\s*(?:biological|per)',
    ]:
        m = _re.search(pattern, text_lower)
        if m:
            n = int(m.group(1))
            if 1 < n <= 20: return n
    return None

def extract_fraction_count(text: str):
    """Extract fraction count from paper text."""
    text_lower = text.lower()
    for pattern in [
        r'(\d+)\s*(?:SCX|high.?pH|RPLC|bRP|SAX|gel)\s+fraction',
        r'(?:fractionat\w+|separated|divided|split)\s+(?:into|to|in)\s+(\d+)',
        r'(\d+)\s*-\s*fraction',
        r'(\d+)\s+fraction\w*\s+(?:were|was|per|from)',
    ]:
        m = _re.search(pattern, text_lower)
        if m:
            n = int(m.group(1))
            if 2 <= n <= 200: return n
    return None

def assign_cyclic(n_rows, n_vals):
    """Create cycling 1..N assignments. Returns 'Not Applicable' if no signal."""
    if n_vals is None:
        return ['Not Applicable'] * n_rows
    return [str((i % n_vals) + 1) for i in range(n_rows)]

# ── Layer 3: Global training majority fallback ───────────────────────
# Pre-computed from 103 training SDRFs: most common non-NA value per column.
# Since test PXDs don't overlap with training PXDs, only global modes apply.
GLOBAL_MAJORITY = {
    'Characteristics[Organism]':       'Homo sapiens',
    'Characteristics[OrganismPart]':   'blood plasma',
    'Characteristics[Disease]':        'not available',
    'Characteristics[CellType]':       'not available',
    'Characteristics[CellLine]':       'not available',
    'Characteristics[Sex]':            'not available',
    'Characteristics[Age]':            'not available',
    'Characteristics[AncestryCategory]': 'not available',
    'Characteristics[DevelopmentalStage]': 'not available',
    'Characteristics[Specimen]':       'not available',
    'Characteristics[MaterialType]':   'tissue',
    'Characteristics[Label]':          'AC=MS:1002038;NT=label free sample',
    'Characteristics[CleavageAgent]':  'AC=MS:1001251;NT=Trypsin',
    'Characteristics[Modification]':   'NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed',
    'Characteristics[Modification].1': 'NT=Oxidation;AC=UNIMOD:35;MT=Variable;TA=M',
    'Characteristics[Modification].2': 'NT=Acetyl;AC=UNIMOD:1;PP=Protein N-term;MT=variable',
    'Characteristics[Modification].3': 'NT= TMT6plex;AC=UNIMOD:737;TA=K;MT=Fixed',
    'Characteristics[Modification].4': 'NT= TMT6plex;AC=UNIMOD:737;PP=Any N-term;MT=Fixed',
    'Characteristics[Modification].5': 'NT=Ammonia-loss;AC=UNIMOD:385;TA=Q,C;MT=Variable',
    'Characteristics[Modification].6': 'NT=iTRAQ;PP=Any N-term;TA=K;AC=UNIMOD:214;MT=fixed',
    'Characteristics[BiologicalReplicate]': '1',
    'Characteristics[TechnicalReplicate]':  '1',
    'Characteristics[ReductionReagent]':    'DTT',
    'Characteristics[AlkylationReagent]':   'IAA',
    'Characteristics[Treatment]':      'none',
    'Characteristics[Strain]':         'A/J',
    'Characteristics[Staining]':       'unstained',
    'Characteristics[SyntheticPeptide]': 'synthetic',
    'Characteristics[Temperature]':    '40',
    'Characteristics[Time]':           '30m',
    'Characteristics[GrowthRate]':     'slow',
    'Characteristics[Bait]':           'EV',
    'Characteristics[Compound]':       'drug immune checkpoint inhibitors',
    'Characteristics[Depletion]':      'depletion',
    'Characteristics[SamplingTime]':   '3',
    'Characteristics[PooledSample]':   'TW1-Pool1-ctrl ApoE3/3 41/42',
    'Characteristics[SpikedCompound]': 'CT=mixture;QY=250 amol;CN=UPS1;CV=Standards Research Group',
    'Characteristics[TumorSize]':      '6.5cm',
    'Characteristics[CellPart]':       'Cytosol',
    'Characteristics[BMI]':            '26.5',
    'Comment[Instrument]':             'NT=Q Exactive;AC=MS:1001911',
    'Comment[FragmentationMethod]':    'AC=MS:1000422;NT=HCD',
    'Comment[PrecursorMassTolerance]': '10 ppm',
    'Comment[FragmentMassTolerance]':  'not available',
    'Comment[FractionIdentifier]':     '1',
    'Comment[FractionationMethod]':    'no fractionation',
    'Comment[MS2MassAnalyzer]':        'AC=MS:1000484; NT=Orbitrap',
    'Comment[Separation]':             'AC=PRIDE:0000563;NT=Reversed-phase chromatography',
    'Comment[CollisionEnergy]':        'not available',
    'Comment[EnrichmentMethod]':       'enrichment of phosphorylated Protein',
    'Comment[NumberOfMissedCleavages]': '1',
    'FactorValue[Bait]':               'EV',
    'FactorValue[CellPart]':           'Cytosol',
    'FactorValue[Compound]':           'drug immune checkpoint inhibitors',
    'FactorValue[Disease]':            'not available',
    'FactorValue[FractionIdentifier]': '1',
    'FactorValue[Temperature]':        '40',
    'FactorValue[Treatment]':          'none',
}

print(f'Rule-based extractors ready.')
print(f'Global majority fallback: {len(GLOBAL_MAJORITY)} columns.')

## 8. Build submission \u2014 LLM \u2192 Rules \u2192 Majority fallback

In [ ]:
# All SDRF columns from SampleSubmission
SDRF_COLS = [c for c in sample_sub.columns
             if c not in ('ID', 'PXD', 'Raw Data File')]

# ── Layer 1 helpers: LLM output parsing ──────────────────────────────

def _detect_format(extracted: dict) -> str:
    if not extracted:
        return 'flat'
    first = next(iter(extracted))
    if first.startswith('Characteristics') or first.startswith('Comment') \
            or first.startswith('FactorValue') or first in ('Source Name', 'Assay Name'):
        return 'flat'
    return 'per_file'

def _coerce_value(val) -> str:
    if val is None:
        return 'Not Applicable'
    if isinstance(val, list):
        parts = [str(v).strip() for v in val if v is not None and str(v).strip()]
        return '; '.join(parts) if parts else 'Not Applicable'
    s = str(val).strip()
    return s if s and s.lower() not in ('', 'none', 'null') else 'Not Applicable'

def llm_get_value(extracted: dict, raw_file: str, col: str) -> str:
    fmt = _detect_format(extracted)
    if fmt == 'flat':
        return _coerce_value(extracted.get(col))
    raw_lower = raw_file.lower()
    file_data = extracted.get(raw_file)
    if file_data is None:
        file_data = next(
            (v for k, v in extracted.items() if k.lower() == raw_lower), {}
        )
    if not isinstance(file_data, dict):
        return 'Not Applicable'
    return _coerce_value(file_data.get(col))

def llm_get_modification_n(extracted: dict, raw_file: str, n: int) -> str:
    fmt = _detect_format(extracted)
    if fmt == 'flat':
        mods = extracted.get('Characteristics[Modification]', [])
    else:
        raw_lower = raw_file.lower()
        file_data = extracted.get(raw_file) or next(
            (v for k, v in extracted.items() if k.lower() == raw_lower), {}
        )
        mods = file_data.get('Characteristics[Modification]', []) if isinstance(file_data, dict) else []
    if isinstance(mods, str):
        mods = [m.strip() for m in mods.split(';') if '=' in m or m.strip()]
    if isinstance(mods, list) and len(mods) > n:
        s = str(mods[n]).strip()
        return s if s and s != 'Not Applicable' else 'Not Applicable'
    return 'Not Applicable'

# ── Layer 2: pre-compute rule-based assignments per PXD ──────────────
pxd_rule_bio_rep = {}
pxd_rule_frac_id = {}

for pxd, raw_files in sorted(pxd_to_raws.items()):
    text = pxd_texts.get(pxd, '')
    n_rows = len(raw_files)
    rep_n = extract_replicate_count(text)
    pxd_rule_bio_rep[pxd] = assign_cyclic(n_rows, rep_n)
    frac_n = extract_fraction_count(text)
    pxd_rule_frac_id[pxd] = assign_cyclic(n_rows, frac_n)
    if rep_n or frac_n:
        print(f'  {pxd}: bio_rep={rep_n}, frac={frac_n} ({n_rows} rows)')

# ── Build submission: LLM -> Rules -> Majority fallback ──────────────
rows = []
pxd_row_idx = {}

for _, sub_row in sample_sub.iterrows():
    row_id   = sub_row['ID']
    pxd      = sub_row['PXD']
    raw_file = sub_row['Raw Data File']
    extracted = pxd_extractions.get(pxd, {})

    idx = pxd_row_idx.get(pxd, 0)
    pxd_row_idx[pxd] = idx + 1

    new_row = {'ID': row_id, 'PXD': pxd, 'Raw Data File': raw_file}

    for col in SDRF_COLS:
        # Layer 1: LLM extraction
        mod_match = re.match(r'Characteristics\[Modification\]\.(\d+)$', col)
        if mod_match:
            val = llm_get_modification_n(extracted, raw_file, int(mod_match.group(1)))
        else:
            val = llm_get_value(extracted, raw_file, col)

        # Layer 2: Rule-based overrides
        if val == 'Not Applicable':
            if col == 'Characteristics[BiologicalReplicate]':
                rule_vals = pxd_rule_bio_rep.get(pxd, [])
                if idx < len(rule_vals):
                    val = rule_vals[idx]
            elif col == 'Comment[FractionIdentifier]':
                rule_vals = pxd_rule_frac_id.get(pxd, [])
                if idx < len(rule_vals):
                    val = rule_vals[idx]

        # Layer 3: Global majority fallback
        if val == 'Not Applicable':
            fallback = GLOBAL_MAJORITY.get(col)
            if fallback:
                val = fallback

        new_row[col] = val

    rows.append(new_row)

submission = pd.DataFrame(rows)
print(f'\nSubmission shape: {submission.shape}')

# Fill-rate report
print('\nFill rates:')
n_total_filled = 0
n_total_cells = 0
for col in SDRF_COLS:
    n_filled = (submission[col] != 'Not Applicable').sum()
    n_total_filled += n_filled
    n_total_cells += len(submission)
    if n_filled > 0:
        pct = n_filled / len(submission) * 100
        marker = ''
        if pct == 100: marker = ' [majority]'
        print(f'  {col:<50} {n_filled:>5} / {len(submission)} ({pct:.1f}%){marker}')

n_empty = sum(1 for c in SDRF_COLS if (submission[c] == 'Not Applicable').all())
print(f'\nOverall: {n_total_filled}/{n_total_cells} cells filled ({n_total_filled/n_total_cells*100:.1f}%)')
print(f'Columns still all empty: {n_empty} / {len(SDRF_COLS)}')

display(submission[['ID','PXD','Raw Data File',
                     'Characteristics[Organism]',
                     'Characteristics[Disease]',
                     'Comment[Instrument]']].head(5))

## 9. Validate and save

In [10]:
# Validate
assert len(submission) == len(sample_sub), \
    f'Row count mismatch: {len(submission)} vs {len(sample_sub)}'
assert (submission['ID'].astype(str).values ==
        sample_sub['ID'].astype(str).values).all(), 'ID mismatch'
assert (submission['PXD'].values ==
        sample_sub['PXD'].values).all(), 'PXD mismatch'
print('Validation passed.')

# Fill-rate report
print('\nFill rates for key columns:')
key_cols = [
    'Characteristics[Organism]', 'Characteristics[Disease]',
    'Characteristics[OrganismPart]', 'Characteristics[Modification]',
    'Characteristics[CleavageAgent]', 'Comment[Instrument]',
    'Comment[FragmentationMethod]', 'Comment[PrecursorMassTolerance]',
]
for col in key_cols:
    if col in submission.columns:
        n = (submission[col] != 'Not Applicable').sum()
        print(f'  {col:<45} {n:>5} / {len(submission)} ({n/len(submission)*100:.1f}%)')

# Save
submission.to_csv(OUT_PATH, index=False)
print(f'\nSaved → {OUT_PATH}')
print('Upload submission_final.csv to Kaggle.')

Validation passed.

Fill rates for key columns:
  Characteristics[Organism]                        42 / 1659 (2.5%)
  Characteristics[Disease]                         30 / 1659 (1.8%)
  Characteristics[OrganismPart]                    98 / 1659 (5.9%)
  Characteristics[Modification]                   104 / 1659 (6.3%)
  Characteristics[CleavageAgent]                   42 / 1659 (2.5%)
  Comment[Instrument]                              98 / 1659 (5.9%)
  Comment[FragmentationMethod]                     98 / 1659 (5.9%)
  Comment[PrecursorMassTolerance]                   0 / 1659 (0.0%)

Saved → c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\outputs\submission_final.csv
Upload submission_final.csv to Kaggle.
